# 02. Mini Photometry Walkthrough

A compact, self-contained illustration of the steps the `prose2` pipeline
runs on every frame: detect sources, place apertures, subtract the sky,
and form a differential flux. We use a small synthetic image so the
notebook needs no FITS files, then connect the result to a real pipeline
lightcurve at the end.

**Environment:** run with the `prose` conda kernel (provides `photutils`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from photutils.detection import DAOStarFinder
from photutils.aperture import (
    CircularAperture,
    CircularAnnulus,
    ApertureStats,
    aperture_photometry,
)

rng = np.random.default_rng(42)

## 1. A synthetic star field

A flat sky background with Poisson-like noise plus a handful of Gaussian
stars. One star is made brighter to act as the science target.

In [ ]:
SIZE = 128
N_STARS = 8
TARGET_IDX = 0

sky = 100.0
image = rng.normal(sky, 5.0, size=(SIZE, SIZE))
yy, xx = np.mgrid[0:SIZE, 0:SIZE]
true_xy = []
for i in range(N_STARS):
    x, y = rng.uniform(12, SIZE - 12, size=2)
    amp = 8000.0 if i == TARGET_IDX else rng.uniform(1500, 5000)
    sigma = rng.uniform(1.6, 2.2)
    image += amp * np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    true_xy.append((x, y))
true_xy = np.array(true_xy)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(image, origin="lower", cmap="gray_r")
fig.colorbar(im, ax=ax, label="counts")
_ = ax.set_title("Synthetic frame")

## 2. Source detection

`DAOStarFinder` finds point sources above a threshold set relative to the
background noise. In the real pipeline this seeds the aperture positions
(refined against a Gaia reference for instruments with WCS).

In [ ]:
bkg = np.median(image)
noise = np.std(image - bkg)
finder = DAOStarFinder(fwhm=4.0, threshold=8 * noise)
sources = finder(image - bkg)
positions = np.transpose((sources["xcentroid"], sources["ycentroid"]))
print(f"Detected {len(sources)} sources")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, origin="lower", cmap="gray_r")
ax.scatter(positions[:, 0], positions[:, 1], s=120, facecolors="none",
           edgecolors="C3", label="detected")
ax.legend(loc="upper right")
_ = ax.set_title("Source detection")

## 3. Aperture photometry

A circular aperture sums each star's light; a surrounding annulus measures
the local sky. The star flux is the aperture sum minus the sky level
scaled to the aperture area. We use the sigma-clipped median sky from
`ApertureStats`, which is robust to neighboring stars in the annulus.

In [ ]:
R_AP, R_IN, R_OUT = 6.0, 10.0, 15.0
apertures = CircularAperture(positions, r=R_AP)
annuli = CircularAnnulus(positions, r_in=R_IN, r_out=R_OUT)

phot = aperture_photometry(image, apertures)
sky_med = ApertureStats(image, annuli).median
phot["sky_per_pix"] = sky_med
phot["net_flux"] = phot["aperture_sum"] - sky_med * apertures.area

for col in ("aperture_sum", "sky_per_pix", "net_flux"):
    phot[col].info.format = "%.1f"
print(phot["id", "xcenter", "ycenter", "aperture_sum", "sky_per_pix", "net_flux"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, origin="lower", cmap="gray_r")
apertures.plot(ax=ax, color="C3", lw=1.2)
annuli.plot(ax=ax, color="C0", lw=1.0)
_ = ax.set_title("Apertures (red) and sky annuli (blue)")

## 4. Differential photometry

Atmosphere and instrument make every star's measured flux wobble together.
Dividing the target by a sum of comparison stars cancels most of that
common-mode signal, leaving the target's intrinsic variations. Over a full
sequence of frames this ratio, one value per frame, becomes the lightcurve.

In [ ]:
target_i = int(np.argmax(sources["flux"]))
comp_mask = np.ones(len(sources), dtype=bool)
comp_mask[target_i] = False

target_flux = float(phot["net_flux"][target_i])
comp_flux = float(np.sum(phot["net_flux"][comp_mask]))
differential = target_flux / comp_flux

print(f"target net flux      : {target_flux:11.1f}")
print(f"sum of comparisons   : {comp_flux:11.1f}")
print(f"differential flux    : {differential:.6f}")

print("\nOn a single frame this is just a number. Repeated over every")
print("frame in the night, target/comparison traces out the lightcurve")
print("you loaded in notebook 01.")

## 5. From concept to a real lightcurve

The pipeline does exactly this, per frame, with careful aperture
optimization and comparison-star selection. The `Flux` column of a prose
CSV is the finished differential lightcurve.

In [ ]:
import os, re
from pathlib import Path
import pandas as pd

prose = Path(os.environ.get("MUSCAT_PROSE_DIR", Path.home() / "ql" / "prose"))
example = prose / "muscat3" / "230521" / "TOI5191_muscat3_rp_230521.csv"
if example.exists():
    lc = pd.read_csv(example)
    precision_ppt = float(lc["Err"].median() * 1e3)
    print(f"{example.name}: {len(lc)} frames, "
          f"median per-point precision {precision_ppt:.2f} ppt")
    fig, ax = plt.subplots(figsize=(8, 4))
    t = (lc["BJD_TDB"] - lc["BJD_TDB"].iloc[0]) * 24
    ax.scatter(t, lc["Flux"], s=6, color="C0")
    ax.set_xlabel("Time from first frame (hours)")
    ax.set_ylabel("Relative flux")
    ax.set_title("Real prose2 differential lightcurve (TOI-5191, rp)")
    fig.tight_layout()
    plt.show()
else:
    print(f"Example CSV not found under {prose}.")
    print("Set MUSCAT_PROSE_DIR to view a real lightcurve here.")

## Summary

- **Detect** sources, **place** apertures, **subtract** the sky, **ratio**
  target to comparisons: that is aperture differential photometry.
- Doing this per frame over a night produces the lightcurve.
- The production pipeline (`prose2`) automates and optimizes every step;
  the `Flux` column you load elsewhere is its output.